<a href="https://colab.research.google.com/github/amzad-786githumb/AIR_LLM_Research/blob/main/05_Dataset_and_Feature_Profiling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [30]:
# ============================================================
# 05.1 ENVIRONMENT AND GOOGLE DRIVE
# ============================================================

from pathlib import Path
import json
import math
import time
import warnings

import numpy as np
import pandas as pd

from scipy.stats import (
    skew,
    kurtosis,
    entropy
)

from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# GOOGLE DRIVE
# ------------------------------------------------------------

from google.colab import drive

drive.mount(
    "/content/drive",
    force_remount=False
)

# ------------------------------------------------------------
# PROJECT PATH
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/AIR_LLM_Research"
)

assert PROJECT_ROOT.exists(), (
    f"Project root not found:\n{PROJECT_ROOT}"
)

# ------------------------------------------------------------
# DATASETS
# ------------------------------------------------------------

DATASET_IDS = [
    "adult_income",
    "bank_marketing",
    "diabetes_130us"
]

print("=" * 90)
print("NOTEBOOK 05 — DATASET AND FEATURE PROFILING")
print("=" * 90)

print(
    f"Project root: {PROJECT_ROOT}"
)

print(
    f"Datasets    : {DATASET_IDS}"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
NOTEBOOK 05 — DATASET AND FEATURE PROFILING
Project root: /content/drive/MyDrive/AIR_LLM_Research
Datasets    : ['adult_income', 'bank_marketing', 'diabetes_130us']


In [31]:
# ============================================================
# 05.2 PROJECT DIRECTORIES
# ============================================================

SPLIT_ROOT = (
    PROJECT_ROOT /
    "data" /
    "splits"
)

PROFILE_ROOT = (
    PROJECT_ROOT /
    "results" /
    "profiling"
)

DATASET_PROFILE_DIR = (
    PROFILE_ROOT /
    "dataset"
)

FEATURE_PROFILE_DIR = (
    PROFILE_ROOT /
    "feature"
)

SUMMARY_DIR = (
    PROFILE_ROOT /
    "summary"
)

for path in [
    PROFILE_ROOT,
    DATASET_PROFILE_DIR,
    FEATURE_PROFILE_DIR,
    SUMMARY_DIR
]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )

print("=" * 90)
print("PROFILING DIRECTORIES READY")
print("=" * 90)

print(
    f"Profile root: {PROFILE_ROOT}"
)

PROFILING DIRECTORIES READY
Profile root: /content/drive/MyDrive/AIR_LLM_Research/results/profiling


In [32]:
# ============================================================
# 05.3 LOAD TRAINING DATA
# ============================================================

TRAINING_DATA = {}

TARGET_REGISTRY = {
    "adult_income": "income",
    "bank_marketing": "y",
    "diabetes_130us": "readmitted"
}

for dataset_id in DATASET_IDS:

    dataset_dir = (
        SPLIT_ROOT /
        dataset_id
    )

    X_path = (
        dataset_dir /
        "X_train.csv"
    )

    y_path = (
        dataset_dir /
        "y_train.csv"
    )

    if not X_path.exists():

        raise FileNotFoundError(
            f"Training features not found:\n{X_path}"
        )

    if not y_path.exists():

        raise FileNotFoundError(
            f"Training target not found:\n{y_path}"
        )

    X_train = pd.read_csv(
        X_path,
        low_memory=False
    )

    y_train = pd.read_csv(
        y_path,
        low_memory=False
    )

    target = TARGET_REGISTRY[
        dataset_id
    ]

    if target in X_train.columns:

        # Safety: target must not be inside X.
        X_train = X_train.drop(
            columns=[target]
        )

    if target not in y_train.columns:

        if y_train.shape[1] == 1:

            y_train.columns = [
                target
            ]

        else:

            raise KeyError(
                f"Target '{target}' "
                f"not found in y_train for "
                f"{dataset_id}"
            )

    TRAINING_DATA[
        dataset_id
    ] = pd.concat(
        [
            X_train.reset_index(drop=True),
            y_train[[target]].reset_index(drop=True)
        ],
        axis=1
    )

print("=" * 90)
print("TRAINING DATA LOADED")
print("=" * 90)

for dataset_id, df in TRAINING_DATA.items():

    print(
        f"{dataset_id:20s} "
        f"{df.shape[0]:,} rows × "
        f"{df.shape[1]:,} columns"
    )

TRAINING DATA LOADED
adult_income         19,522 rows × 15 columns
bank_marketing       27,126 rows × 17 columns
diabetes_130us       61,059 rows × 48 columns


In [33]:
# ============================================================
# 05.4 TRAINING DATA VALIDATION
# ============================================================

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    assert target in df.columns

    assert df.shape[0] > 0

    assert df.shape[1] > 1

    assert not df.columns.duplicated().any()

    print(
        f"{dataset_id:20s} "
        f"PASS | target={target} | "
        f"shape={df.shape}"
    )

print()
print("Training-data validation completed.")

adult_income         PASS | target=income | shape=(19522, 15)
bank_marketing       PASS | target=y | shape=(27126, 17)
diabetes_130us       PASS | target=readmitted | shape=(61059, 48)

Training-data validation completed.


In [34]:
# ============================================================
# 05.5 DATASET SIZE
# ============================================================

DATASET_SIZE_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    DATASET_SIZE_PROFILE.append({

        "dataset_id":
            dataset_id,

        "n_rows":
            int(df.shape[0]),

        "n_columns":
            int(df.shape[1]),

        "memory_mb":
            float(
                df.memory_usage(
                    deep=True
                ).sum()
                / (1024 ** 2)
            )
    })

DATASET_SIZE_DF = pd.DataFrame(
    DATASET_SIZE_PROFILE
)

display(
    DATASET_SIZE_DF
)

,dataset_id,n_rows,n_columns,memory_mb
0,adult_income,19522,15,10.580482
1,bank_marketing,27126,17,15.448343
2,diabetes_130us,61059,48,112.802345


In [35]:
# ============================================================
# 05.6 NUMBER OF FEATURES
# ============================================================

FEATURE_COUNT_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    features = [
        c for c in df.columns
        if c != target
    ]

    FEATURE_COUNT_PROFILE.append({

        "dataset_id":
            dataset_id,

        "n_features":
            int(len(features)),

        "n_numeric_features":
            int(
                df[features]
                .select_dtypes(
                    include=np.number
                )
                .shape[1]
            ),

        "n_categorical_features":
            int(
                df[features]
                .select_dtypes(
                    exclude=np.number
                )
                .shape[1]
            )
    })

FEATURE_COUNT_DF = pd.DataFrame(
    FEATURE_COUNT_PROFILE
)

display(
    FEATURE_COUNT_DF
)

,dataset_id,n_features,n_numeric_features,n_categorical_features
0,adult_income,14,6,8
1,bank_marketing,16,7,9
2,diabetes_130us,47,11,36


In [36]:
# ============================================================
# 05.7 NUMERICAL / CATEGORICAL RATIO
# ============================================================

TYPE_RATIO_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    X = df.drop(
        columns=[target]
    )

    n_numeric = len(
        X.select_dtypes(
            include=np.number
        ).columns
    )

    n_categorical = (
        X.shape[1] -
        n_numeric
    )

    total = X.shape[1]

    TYPE_RATIO_PROFILE.append({

        "dataset_id":
            dataset_id,

        "numeric_count":
            int(n_numeric),

        "categorical_count":
            int(n_categorical),

        "numeric_ratio":
            float(
                n_numeric / total
            ) if total else 0.0,

        "categorical_ratio":
            float(
                n_categorical / total
            ) if total else 0.0
    })

TYPE_RATIO_DF = pd.DataFrame(
    TYPE_RATIO_PROFILE
)

display(
    TYPE_RATIO_DF
)

,dataset_id,numeric_count,categorical_count,numeric_ratio,categorical_ratio
0,adult_income,6,8,0.428571,0.571429
1,bank_marketing,7,9,0.437500,0.562500
2,diabetes_130us,11,36,0.234043,0.765957


In [37]:
# ============================================================
# 05.8 CLASS DISTRIBUTION
# ============================================================

CLASS_DISTRIBUTIONS = {}

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    counts = (
        df[target]
        .value_counts(
            dropna=False
        )
    )

    proportions = (
        df[target]
        .value_counts(
            normalize=True,
            dropna=False
        )
    )

    records = []

    for value in counts.index:

        records.append({

            "dataset_id":
                dataset_id,

            "target":
                target,

            "class":
                str(value),

            "count":
                int(counts.loc[value]),

            "proportion":
                float(
                    proportions.loc[value]
                )
        })

    CLASS_DISTRIBUTIONS[
        dataset_id
    ] = pd.DataFrame(
        records
    )

CLASS_DISTRIBUTION_DF = pd.concat(
    CLASS_DISTRIBUTIONS.values(),
    ignore_index=True
)

display(
    CLASS_DISTRIBUTION_DF
)

,dataset_id,target,class,count,proportion
0,adult_income,income,<=50K,14819,0.759092
1,adult_income,income,>50K,4703,0.240908
2,bank_marketing,y,no,23953,0.883027
3,bank_marketing,y,yes,3173,0.116973
4,diabetes_130us,readmitted,NO,32918,0.539118
5,diabetes_130us,readmitted,>30,21327,0.349285
6,diabetes_130us,readmitted,<30,6814,0.111597


In [38]:
# ============================================================
# 05.9 MISSINGNESS STATISTICS
# ============================================================

MISSINGNESS_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for column in df.columns:

        missing_count = int(
            df[column].isna().sum()
        )

        missing_rate = float(
            df[column].isna().mean()
        )

        MISSINGNESS_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "missing_count":
                missing_count,

            "missing_rate":
                missing_rate,

            "is_target":
                column == target
        })

MISSINGNESS_DF = pd.DataFrame(
    MISSINGNESS_PROFILE
)

display(
    MISSINGNESS_DF.head(20)
)

,dataset_id,feature,missing_count,missing_rate,is_target
0,adult_income,age,0,0.0,False
1,adult_income,workclass,0,0.0,False
2,adult_income,fnlwgt,0,0.0,False
3,adult_income,education,0,0.0,False
4,adult_income,education_num,0,0.0,False
5,adult_income,marital_status,0,0.0,False
6,adult_income,occupation,0,0.0,False
7,adult_income,relationship,0,0.0,False
8,adult_income,race,0,0.0,False
9,adult_income,sex,0,0.0,False


In [39]:
# ============================================================
# 05.10 CARDINALITY
# ============================================================

CARDINALITY_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for column in df.columns:

        non_missing = df[column].dropna()

        n_unique = int(
            non_missing.nunique()
        )

        n_rows = len(df)

        cardinality_ratio = (
            n_unique / n_rows
            if n_rows > 0
            else 0.0
        )

        CARDINALITY_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "unique_count":
                n_unique,

            "cardinality_ratio":
                float(
                    cardinality_ratio
                ),

            "is_constant":
                n_unique <= 1,

            "is_high_cardinality":
                cardinality_ratio >= 0.50
        })

CARDINALITY_DF = pd.DataFrame(
    CARDINALITY_PROFILE
)

display(
    CARDINALITY_DF.head(20)
)

,dataset_id,feature,unique_count,cardinality_ratio,is_constant,is_high_cardinality
0,adult_income,age,73,0.003739,False,False
1,adult_income,workclass,9,0.000461,False,False
2,adult_income,fnlwgt,14799,0.758068,False,True
3,adult_income,education,16,0.000820,False,False
4,adult_income,education_num,16,0.000820,False,False
5,adult_income,marital_status,7,0.000359,False,False
6,adult_income,occupation,15,0.000768,False,False
7,adult_income,relationship,6,0.000307,False,False
8,adult_income,race,5,0.000256,False,False
9,adult_income,sex,2,0.000102,False,False


In [40]:
# ============================================================
# 05.11 DISTRIBUTION STATISTICS
# ============================================================

DISTRIBUTION_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    numeric_columns = (
        df.drop(columns=[target])
        .select_dtypes(
            include=np.number
        )
        .columns
    )

    for column in numeric_columns:

        series = (
            pd.to_numeric(
                df[column],
                errors="coerce"
            )
            .dropna()
        )

        if series.empty:
            continue

        mean_value = float(
            series.mean()
        )

        median_value = float(
            series.median()
        )

        std_value = float(
            series.std()
        )

        min_value = float(
            series.min()
        )

        max_value = float(
            series.max()
        )

        skew_value = float(
            series.skew()
        )

        kurt_value = float(
            series.kurt()
        )

        DISTRIBUTION_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "mean":
                mean_value,

            "median":
                median_value,

            "std":
                std_value,

            "min":
                min_value,

            "max":
                max_value,

            "skewness":
                skew_value,

            "kurtosis":
                kurt_value,

            "distribution_shape":
                (
                    "right_skewed"
                    if skew_value > 1
                    else
                    "left_skewed"
                    if skew_value < -1
                    else
                    "approximately_symmetric"
                )
        })

DISTRIBUTION_DF = pd.DataFrame(
    DISTRIBUTION_PROFILE
)

display(
    DISTRIBUTION_DF.head(20)
)

,dataset_id,feature,mean,median,std,min,max,skewness,kurtosis,distribution_shape
0,adult_income,age,38.539852,37.0,13.645484,17.0,90.0,0.558961,-0.164794,approximately_symmetric
1,adult_income,fnlwgt,190020.295205,178319.0,106163.578269,12285.0,1484705.0,1.430776,5.829256,right_skewed
2,adult_income,education_num,10.083752,10.0,2.567326,1.0,16.0,-0.297397,0.588060,approximately_symmetric
3,adult_income,capital_gain,1061.727948,0.0,7270.406604,0.0,99999.0,12.104584,159.283980,right_skewed
4,adult_income,capital_loss,87.941860,0.0,404.021716,0.0,4356.0,4.569990,20.109857,right_skewed
5,adult_income,hours_per_week,40.435867,40.0,12.401671,1.0,99.0,0.220144,2.931302,approximately_symmetric
6,bank_marketing,age,40.852393,39.0,10.636893,18.0,95.0,0.706297,0.402527,approximately_symmetric
7,bank_marketing,balance,1361.175035,449.0,3053.628721,-8019.0,102127.0,8.613740,147.921643,right_skewed
8,bank_marketing,day,15.781132,16.0,8.320549,1.0,31.0,0.098508,-1.061187,approximately_symmetric
9,bank_marketing,duration,258.993770,180.0,260.659786,0.0,4918.0,3.278843,20.597327,right_skewed


In [41]:
# ============================================================
# 05.12 OUTLIER STATISTICS
# ============================================================

OUTLIER_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    numeric_columns = (
        df.drop(columns=[target])
        .select_dtypes(
            include=np.number
        )
        .columns
    )

    for column in numeric_columns:

        series = (
            pd.to_numeric(
                df[column],
                errors="coerce"
            )
            .dropna()
        )

        if series.empty:
            continue

        q1 = float(
            series.quantile(0.25)
        )

        q3 = float(
            series.quantile(0.75)
        )

        iqr = q3 - q1

        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_mask = (
            (series < lower) |
            (series > upper)
        )

        outlier_count = int(
            outlier_mask.sum()
        )

        OUTLIER_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "q1":
                q1,

            "q3":
                q3,

            "iqr":
                float(iqr),

            "lower_bound":
                float(lower),

            "upper_bound":
                float(upper),

            "outlier_count":
                outlier_count,

            "outlier_rate":
                float(
                    outlier_count /
                    len(series)
                )
        })

OUTLIER_DF = pd.DataFrame(
    OUTLIER_PROFILE
)

display(
    OUTLIER_DF.head(20)
)

,dataset_id,feature,q1,q3,iqr,lower_bound,upper_bound,outlier_count,outlier_rate
0,adult_income,age,28.0,47.00,19.00,-0.500,75.500,138,0.007069
1,adult_income,fnlwgt,117629.5,237047.00,119417.50,-61496.750,416173.250,605,0.030991
2,adult_income,education_num,9.0,12.00,3.00,4.500,16.500,710,0.036369
3,adult_income,capital_gain,0.0,0.00,0.00,0.000,0.000,1633,0.083649
4,adult_income,capital_loss,0.0,0.00,0.00,0.000,0.000,917,0.046973
5,adult_income,hours_per_week,40.0,45.00,5.00,32.500,52.500,5416,0.277431
6,bank_marketing,age,33.0,48.00,15.00,10.500,70.500,297,0.010949
7,bank_marketing,balance,74.0,1425.00,1351.00,-1952.500,3451.500,2866,0.105655
8,bank_marketing,day,8.0,21.00,13.00,-11.500,40.500,0,0.000000
9,bank_marketing,duration,103.0,317.75,214.75,-219.125,639.875,2009,0.074062


In [42]:
# ============================================================
# 05.13 CORRELATIONS
# ============================================================

CORRELATION_PROFILE = {}

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    numeric_df = (
        df.drop(columns=[target])
        .select_dtypes(
            include=np.number
        )
    )

    if numeric_df.shape[1] >= 2:

        CORRELATION_PROFILE[
            dataset_id
        ] = numeric_df.corr(
            method="spearman"
        )

    else:

        CORRELATION_PROFILE[
            dataset_id
        ] = pd.DataFrame()

print(
    "Spearman correlation matrices constructed."
)

Spearman correlation matrices constructed.


In [43]:
# ============================================================
# 05.14 MUTUAL INFORMATION
# ============================================================

MI_SAMPLE_SIZE = 20000
MI_RANDOM_STATE = 42

MUTUAL_INFORMATION_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    print(
        f"Computing mutual information: "
        f"{dataset_id}"
    )

    target = TARGET_REGISTRY[
        dataset_id
    ]

    X = df.drop(
        columns=[target]
    ).copy()

    y = df[target].copy()

    if len(X) > MI_SAMPLE_SIZE:

        sample_idx = (
            X.sample(
                n=MI_SAMPLE_SIZE,
                random_state=MI_RANDOM_STATE
            ).index
        )

        X = X.loc[
            sample_idx
        ]

        y = y.loc[
            sample_idx
        ]

    # --------------------------------------------------------
    # Convert features to numeric representation
    # --------------------------------------------------------

    X_encoded = pd.DataFrame(
        index=X.index
    )

    discrete_features = []

    for column in X.columns:

        series = X[column]

        if pd.api.types.is_numeric_dtype(
            series
        ):

            values = (
                pd.to_numeric(
                    series,
                    errors="coerce"
                )
            )

            fill_value = (
                values.median()
                if not values.dropna().empty
                else 0.0
            )

            X_encoded[column] = (
                values.fillna(
                    fill_value
                )
            )

            discrete_features.append(
                False
            )

        else:

            encoded = (
                series.astype("string")
                .fillna("__MISSING__")
                .astype("category")
                .cat.codes
            )

            X_encoded[column] = (
                encoded.astype(float)
            )

            discrete_features.append(
                True
            )

    # --------------------------------------------------------
    # Encode target
    # --------------------------------------------------------

    y_encoded = (
        y.astype("string")
        .fillna("__MISSING__")
        .astype("category")
        .cat.codes
        .to_numpy()
    )

    # --------------------------------------------------------
    # Mutual information
    # --------------------------------------------------------

    mi_values = mutual_info_classif(
        X_encoded.to_numpy(),
        y_encoded,
        discrete_features=
            discrete_features,
        random_state=MI_RANDOM_STATE
    )

    for column, mi in zip(
        X.columns,
        mi_values
    ):

        MUTUAL_INFORMATION_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "mutual_information":
                float(mi)
        })

MUTUAL_INFORMATION_DF = pd.DataFrame(
    MUTUAL_INFORMATION_PROFILE
)

display(
    MUTUAL_INFORMATION_DF.head(20)
)

Computing mutual information: adult_income
Computing mutual information: bank_marketing
Computing mutual information: diabetes_130us


,dataset_id,feature,mutual_information
0,adult_income,age,0.072244
1,adult_income,workclass,0.015795
2,adult_income,fnlwgt,0.025190
3,adult_income,education,0.065343
4,adult_income,education_num,0.067522
5,adult_income,marital_status,0.107862
6,adult_income,occupation,0.065347
7,adult_income,relationship,0.115568
8,adult_income,race,0.005956
9,adult_income,sex,0.027358


In [44]:
# ============================================================
# 05.15 FEATURE DEPENDENCIES
# ============================================================

DEPENDENCY_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    features = [
        c for c in df.columns
        if c != target
    ]

    correlation_matrix = (
        CORRELATION_PROFILE.get(
            dataset_id,
            pd.DataFrame()
        )
    )

    for feature in features:

        max_abs_correlation = 0.0
        strongest_correlated_feature = None

        if (
            not correlation_matrix.empty
            and feature in correlation_matrix.columns
        ):

            correlations = (
                correlation_matrix[
                    feature
                ]
                .drop(labels=[feature], errors="ignore")
                .abs()
                .dropna()
            )

            if not correlations.empty:

                strongest_correlated_feature = (
                    correlations.idxmax()
                )

                max_abs_correlation = float(
                    correlations.max()
                )

        mi_row = MUTUAL_INFORMATION_DF[
            (
                MUTUAL_INFORMATION_DF[
                    "dataset_id"
                ] == dataset_id
            )
            &
            (
                MUTUAL_INFORMATION_DF[
                    "feature"
                ] == feature
            )
        ]

        mutual_information = (
            float(
                mi_row[
                    "mutual_information"
                ].iloc[0]
            )
            if not mi_row.empty
            else 0.0
        )

        DEPENDENCY_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                feature,

            "max_abs_spearman":
                max_abs_correlation,

            "strongest_correlated_feature":
                strongest_correlated_feature,

            "mutual_information":
                mutual_information
        })

DEPENDENCY_DF = pd.DataFrame(
    DEPENDENCY_PROFILE
)

display(
    DEPENDENCY_DF.head(20)
)

,dataset_id,feature,max_abs_spearman,strongest_correlated_feature,mutual_information
0,adult_income,age,0.141710,hours_per_week,0.072244
1,adult_income,workclass,0.000000,None,0.015795
2,adult_income,fnlwgt,0.076763,age,0.025190
3,adult_income,education,0.000000,None,0.065343
4,adult_income,education_num,0.171823,hours_per_week,0.067522
5,adult_income,marital_status,0.000000,None,0.107862
6,adult_income,occupation,0.000000,None,0.065347
7,adult_income,relationship,0.000000,None,0.115568
8,adult_income,race,0.000000,None,0.005956
9,adult_income,sex,0.000000,None,0.027358


In [45]:
# ============================================================
# 05.16 COMPUTATIONAL SCALE
# ============================================================

COMPUTATIONAL_SCALE_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    n_rows = len(df)
    n_features = len(df.columns) - 1

    missing_cells = int(
        df.drop(columns=[target])
        .isna()
        .sum()
        .sum()
    )

    total_cells = (
        n_rows *
        n_features
    )

    COMPUTATIONAL_SCALE_PROFILE.append({

        "dataset_id":
            dataset_id,

        "rows":
            int(n_rows),

        "features":
            int(n_features),

        "cells":
            int(total_cells),

        "missing_cells":
            missing_cells,

        "missing_cell_rate":
            float(
                missing_cells /
                total_cells
            ) if total_cells else 0.0,

        "memory_mb":
            float(
                df.memory_usage(
                    deep=True
                ).sum()
                / (1024 ** 2)
            ),

        "scale_category":
            (
                "small"
                if total_cells < 1_000_000
                else
                "medium"
                if total_cells < 10_000_000
                else
                "large"
            )
    })

COMPUTATIONAL_SCALE_DF = pd.DataFrame(
    COMPUTATIONAL_SCALE_PROFILE
)

display(
    COMPUTATIONAL_SCALE_DF
)

,dataset_id,rows,features,cells,missing_cells,missing_cell_rate,memory_mb,scale_category
0,adult_income,19522,14,273308,0,0.00000,10.580482,small
1,bank_marketing,27126,16,434016,0,0.00000,15.448343,small
2,diabetes_130us,61059,47,2869773,224503,0.07823,112.802345,medium


In [46]:
# ============================================================
# 05.17 DATASET PROFILE CONSTRUCTION
# ============================================================

DATASET_PROFILES = {}

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    X = df.drop(
        columns=[target]
    )

    size_row = DATASET_SIZE_DF[
        DATASET_SIZE_DF[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]

    type_row = TYPE_RATIO_DF[
        TYPE_RATIO_DF[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]

    scale_row = COMPUTATIONAL_SCALE_DF[
        COMPUTATIONAL_SCALE_DF[
            "dataset_id"
        ] == dataset_id
    ].iloc[0]

    dataset_missing_rate = float(
        X.isna().mean().mean()
    )

    target_distribution = (
        CLASS_DISTRIBUTION_DF[
            CLASS_DISTRIBUTION_DF[
                "dataset_id"
            ] == dataset_id
        ]
        .to_dict(
            orient="records"
        )
    )

    DATASET_PROFILES[
        dataset_id
    ] = {

        "dataset_id":
            dataset_id,

        "size": {

            "rows":
                int(size_row["n_rows"]),

            "columns":
                int(size_row["n_columns"]),

            "memory_mb":
                float(size_row["memory_mb"])
        },

        "type": {

            "n_features":
                int(X.shape[1]),

            "numeric_count":
                int(type_row["numeric_count"]),

            "categorical_count":
                int(type_row["categorical_count"]),

            "numeric_ratio":
                float(type_row["numeric_ratio"]),

            "categorical_ratio":
                float(type_row["categorical_ratio"])
        },

        "distribution": {

            "numeric_features":
                DISTRIBUTION_DF[
                    DISTRIBUTION_DF[
                        "dataset_id"
                    ] == dataset_id
                ].to_dict(
                    orient="records"
                )
        },

        "dependency": {

            "feature_dependencies":
                DEPENDENCY_DF[
                    DEPENDENCY_DF[
                        "dataset_id"
                    ] == dataset_id
                ].to_dict(
                    orient="records"
                )
        },

        "missingness": {

            "overall_feature_missing_rate":
                dataset_missing_rate,

            "feature_statistics":
                MISSINGNESS_DF[
                    MISSINGNESS_DF[
                        "dataset_id"
                    ] == dataset_id
                ].to_dict(
                    orient="records"
                )
        },

        "task": {

            "target":
                target,

            "target_distribution":
                target_distribution
        },

        "computational_scale":
            scale_row.to_dict()
    }

print(
    "Dataset profiles constructed."
)

Dataset profiles constructed.


In [47]:
# ============================================================
# 05.18 FEATURE TYPE
# ============================================================

FEATURE_TYPE_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for column in df.columns:

        if column == target:

            feature_role = "target"

        elif pd.api.types.is_numeric_dtype(
            df[column]
        ):

            feature_role = "numerical"

        else:

            feature_role = "categorical"

        FEATURE_TYPE_PROFILE.append({

            "dataset_id":
                dataset_id,

            "feature":
                column,

            "feature_role":
                feature_role,

            "dtype":
                str(df[column].dtype),

            "is_numeric":
                feature_role == "numerical",

            "is_categorical":
                feature_role == "categorical",

            "is_target":
                feature_role == "target"
        })

FEATURE_TYPE_DF = pd.DataFrame(
    FEATURE_TYPE_PROFILE
)

display(
    FEATURE_TYPE_DF.head(20)
)

,dataset_id,feature,feature_role,dtype,is_numeric,is_categorical,is_target
0,adult_income,age,numerical,int64,True,False,False
1,adult_income,workclass,categorical,object,False,True,False
2,adult_income,fnlwgt,numerical,int64,True,False,False
3,adult_income,education,categorical,object,False,True,False
4,adult_income,education_num,numerical,int64,True,False,False
5,adult_income,marital_status,categorical,object,False,True,False
6,adult_income,occupation,categorical,object,False,True,False
7,adult_income,relationship,categorical,object,False,True,False
8,adult_income,race,categorical,object,False,True,False
9,adult_income,sex,categorical,object,False,True,False


In [48]:
# ============================================================
# 05.19 FEATURE MISSINGNESS
# ============================================================

FEATURE_MISSINGNESS_DF = (
    MISSINGNESS_DF.copy()
)

FEATURE_MISSINGNESS_DF[
    "missingness_category"
] = pd.cut(
    FEATURE_MISSINGNESS_DF[
        "missing_rate"
    ],
    bins=[
        -np.inf,
        0.0,
        0.10,
        0.30,
        0.50,
        np.inf
    ],
    labels=[
        "none",
        "low",
        "moderate",
        "high",
        "very_high"
    ]
)

display(
    FEATURE_MISSINGNESS_DF.head(20)
)

,dataset_id,feature,missing_count,missing_rate,is_target,missingness_category
0,adult_income,age,0,0.0,False,none
1,adult_income,workclass,0,0.0,False,none
2,adult_income,fnlwgt,0,0.0,False,none
3,adult_income,education,0,0.0,False,none
4,adult_income,education_num,0,0.0,False,none
5,adult_income,marital_status,0,0.0,False,none
6,adult_income,occupation,0,0.0,False,none
7,adult_income,relationship,0,0.0,False,none
8,adult_income,race,0,0.0,False,none
9,adult_income,sex,0,0.0,False,none


In [49]:
# ============================================================
# 05.20 FEATURE DISTRIBUTION
# ============================================================

FEATURE_DISTRIBUTION_PROFILE = []

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    for column in df.columns:

        if column == target:
            continue

        series = df[column]

        if pd.api.types.is_numeric_dtype(
            series
        ):

            values = (
                pd.to_numeric(
                    series,
                    errors="coerce"
                )
                .dropna()
            )

            if values.empty:

                continue

            skew_value = float(
                values.skew()
            )

            FEATURE_DISTRIBUTION_PROFILE.append({

                "dataset_id":
                    dataset_id,

                "feature":
                    column,

                "distribution_type":
                    "numerical",

                "mean":
                    float(values.mean()),

                "median":
                    float(values.median()),

                "std":
                    float(values.std()),

                "skewness":
                    skew_value,

                "kurtosis":
                    float(values.kurt()),

                "distribution_shape":
                    (
                        "right_skewed"
                        if skew_value > 1
                        else
                        "left_skewed"
                        if skew_value < -1
                        else
                        "approximately_symmetric"
                    )
            })

        else:

            counts = (
                series
                .value_counts(
                    normalize=True,
                    dropna=False
                )
            )

            top_frequency = (
                float(counts.iloc[0])
                if not counts.empty
                else 0.0
            )

            FEATURE_DISTRIBUTION_PROFILE.append({

                "dataset_id":
                    dataset_id,

                "feature":
                    column,

                "distribution_type":
                    "categorical",

                "mean":
                    np.nan,

                "median":
                    np.nan,

                "std":
                    np.nan,

                "skewness":
                    np.nan,

                "kurtosis":
                    np.nan,

                "distribution_shape":
                    "categorical",

                "top_category_frequency":
                    top_frequency
            })

FEATURE_DISTRIBUTION_DF = pd.DataFrame(
    FEATURE_DISTRIBUTION_PROFILE
)

display(
    FEATURE_DISTRIBUTION_DF.head(20)
)

,dataset_id,feature,distribution_type,mean,median,std,skewness,kurtosis,distribution_shape,top_category_frequency
0,adult_income,age,numerical,38.539852,37.0,13.645484,0.558961,-0.164794,approximately_symmetric,NaN
1,adult_income,workclass,categorical,NaN,NaN,NaN,NaN,NaN,categorical,0.693576
2,adult_income,fnlwgt,numerical,190020.295205,178319.0,106163.578269,1.430776,5.829256,right_skewed,NaN
3,adult_income,education,categorical,NaN,NaN,NaN,NaN,NaN,categorical,0.323328
4,adult_income,education_num,numerical,10.083752,10.0,2.567326,-0.297397,0.588060,approximately_symmetric,NaN
5,adult_income,marital_status,categorical,NaN,NaN,NaN,NaN,NaN,categorical,0.459379
6,adult_income,occupation,categorical,NaN,NaN,NaN,NaN,NaN,categorical,0.127753
7,adult_income,relationship,categorical,NaN,NaN,NaN,NaN,NaN,categorical,0.404979
8,adult_income,race,categorical,NaN,NaN,NaN,NaN,NaN,categorical,0.856879
9,adult_income,sex,categorical,NaN,NaN,NaN,NaN,NaN,categorical,0.668528


In [50]:
# ============================================================
# 05.22 FEATURE DEPENDENCIES
# ============================================================

FEATURE_DEPENDENCY_DF = (
    DEPENDENCY_DF.copy()
)

FEATURE_DEPENDENCY_DF[
    "dependency_strength"
] = np.select(

    [
        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ] >= 0.70,

        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ] >= 0.40,

        FEATURE_DEPENDENCY_DF[
            "max_abs_spearman"
        ] >= 0.20
    ],

    [
        "strong",
        "moderate",
        "weak"
    ],

    default="very_weak"
)

display(
    FEATURE_DEPENDENCY_DF.head(20)
)

,dataset_id,feature,max_abs_spearman,strongest_correlated_feature,mutual_information,dependency_strength
0,adult_income,age,0.141710,hours_per_week,0.072244,very_weak
1,adult_income,workclass,0.000000,None,0.015795,very_weak
2,adult_income,fnlwgt,0.076763,age,0.025190,very_weak
3,adult_income,education,0.000000,None,0.065343,very_weak
4,adult_income,education_num,0.171823,hours_per_week,0.067522,very_weak
5,adult_income,marital_status,0.000000,None,0.107862,very_weak
6,adult_income,occupation,0.000000,None,0.065347,very_weak
7,adult_income,relationship,0.000000,None,0.115568,very_weak
8,adult_income,race,0.000000,None,0.005956,very_weak
9,adult_income,sex,0.000000,None,0.027358,very_weak


In [51]:
# ============================================================
# 05.24 FEATURE PROFILE CONSTRUCTION
# ============================================================

FEATURE_PROFILES = {}

for dataset_id, df in TRAINING_DATA.items():

    target = TARGET_REGISTRY[
        dataset_id
    ]

    FEATURE_PROFILES[
        dataset_id
    ] = {}

    for feature in df.columns:

        type_row = FEATURE_TYPE_DF[
            (
                FEATURE_TYPE_DF[
                    "dataset_id"
                ] == dataset_id
            )
            &
            (
                FEATURE_TYPE_DF[
                    "feature"
                ] == feature
            )
        ]

        missing_row = FEATURE_MISSINGNESS_DF[
            (
                FEATURE_MISSINGNESS_DF[
                    "dataset_id"
                ] == dataset_id
            )
            &
            (
                FEATURE_MISSINGNESS_DF[
                    "feature"
                ] == feature
            )
        ]

        distribution_row = (
            FEATURE_DISTRIBUTION_DF[
                (
                    FEATURE_DISTRIBUTION_DF[
                        "dataset_id"
                    ] == dataset_id
                )
                &
                (
                    FEATURE_DISTRIBUTION_DF[
                        "feature"
                    ] == feature
                )
            ]
        )

        cardinality_row = (
            CARDINALITY_DF[
                (
                    CARDINALITY_DF[
                        "dataset_id"
                    ] == dataset_id
                )
                &
                (
                    CARDINALITY_DF[
                        "feature"
                    ] == feature
                )
            ]
        )

        dependency_row = (
            FEATURE_DEPENDENCY_DF[
                (
                    FEATURE_DEPENDENCY_DF[
                        "dataset_id"
                    ] == dataset_id
                )
                &
                (
                    FEATURE_DEPENDENCY_DF[
                        "feature"
                    ] == feature
                )
            ]
        )

        relevance_row = (
            MUTUAL_INFORMATION_DF[ # Changed from FEATURE_PREDICTIVE_RELEVANCE_DF
                (
                    MUTUAL_INFORMATION_DF[ # Changed from FEATURE_PREDICTIVE_RELEVANCE_DF
                        "dataset_id"
                    ] == dataset_id
                )
                &
                (
                    MUTUAL_INFORMATION_DF[ # Changed from FEATURE_PREDICTIVE_RELEVANCE_DF
                        "feature"
                    ] == feature
                )
            ]
        )

        FEATURE_PROFILES[
            dataset_id
        ][
            feature
        ] = {

            "T": (
                type_row.iloc[0].to_dict()
                if not type_row.empty
                else {}
            ),

            "M": (
                missing_row.iloc[0].to_dict()
                if not missing_row.empty
                else {}
            ),

            "D": (
                distribution_row.iloc[0].to_dict()
                if not distribution_row.empty
                else {}
            ),

            "R": (
                relevance_row.iloc[0].to_dict()
                if not relevance_row.empty
                else {}
            ),

            "C": (
                cardinality_row.iloc[0].to_dict()
                if not cardinality_row.empty
                else {}
            ),

            "Y": (
                dependency_row.iloc[0].to_dict()
                if not dependency_row.empty
                else {}
            )
        }

print(
    "Feature profiles constructed."
)

Feature profiles constructed.


In [52]:
# ============================================================
# 05.25 AIR-LLM CONTEXT CONSTRUCTION
# ============================================================

AIR_LLM_CONTEXT = {}

for dataset_id in DATASET_IDS:

    AIR_LLM_CONTEXT[
        dataset_id
    ] = {

        "dataset_profile":
            DATASET_PROFILES[
                dataset_id
            ],

        "feature_profiles":
            FEATURE_PROFILES[
                dataset_id
            ]
    }

print("=" * 90)
print("AIR-LLM CONTEXT CONSTRUCTION COMPLETE")
print("=" * 90)

for dataset_id in DATASET_IDS:

    print(
        f"{dataset_id:20s} | "
        f"features: "
        f"{len(AIR_LLM_CONTEXT[dataset_id]['feature_profiles'])}"
    )

AIR-LLM CONTEXT CONSTRUCTION COMPLETE
adult_income         | features: 15
bank_marketing       | features: 17
diabetes_130us       | features: 48


In [53]:
# ============================================================
# 05.26 COMPACT AIR-LLM CONTEXT
# ============================================================

COMPACT_AIR_LLM_CONTEXT = {}

for dataset_id in DATASET_IDS:

    dataset_profile = (
        DATASET_PROFILES[
            dataset_id
        ]
    )

    feature_profiles = (
        FEATURE_PROFILES[
            dataset_id
        ]
    )

    incomplete_features = {}

    for feature, profile in feature_profiles.items():

        missing_rate = (
            profile
            .get("M", {})
            .get("missing_rate", 0.0)
        )

        if (
            pd.notna(missing_rate)
            and float(missing_rate) > 0
        ):

            incomplete_features[
                feature
            ] = profile

    COMPACT_AIR_LLM_CONTEXT[
        dataset_id
    ] = {

        "dataset": {

            "size":
                dataset_profile["size"],

            "type":
                dataset_profile["type"],

            "missingness":
                dataset_profile[
                    "missingness"
                ],

            "task":
                dataset_profile["task"],

            "computational_scale":
                dataset_profile[
                    "computational_scale"
                ]
        },

        "incomplete_features":
            incomplete_features
    }

print(
    "Compact AIR-LLM contexts constructed."
)

Compact AIR-LLM contexts constructed.


In [54]:
# ============================================================
# 05.27 SAVE DATASET PROFILES
# ============================================================

DATASET_SIZE_DF.to_csv(
    DATASET_PROFILE_DIR /
    "dataset_size.csv",
    index=False
)

FEATURE_COUNT_DF.to_csv(
    DATASET_PROFILE_DIR /
    "feature_counts.csv",
    index=False
)

TYPE_RATIO_DF.to_csv(
    DATASET_PROFILE_DIR /
    "feature_type_ratios.csv",
    index=False
)

CLASS_DISTRIBUTION_DF.to_csv(
    DATASET_PROFILE_DIR /
    "class_distribution.csv",
    index=False
)

MISSINGNESS_DF.to_csv(
    DATASET_PROFILE_DIR /
    "missingness_statistics.csv",
    index=False
)

CARDINALITY_DF.to_csv(
    DATASET_PROFILE_DIR /
    "cardinality_statistics.csv",
    index=False
)

DISTRIBUTION_DF.to_csv(
    DATASET_PROFILE_DIR /
    "distribution_statistics.csv",
    index=False
)

OUTLIER_DF.to_csv(
    DATASET_PROFILE_DIR /
    "outlier_statistics.csv",
    index=False
)

COMPUTATIONAL_SCALE_DF.to_csv(
    DATASET_PROFILE_DIR /
    "computational_scale.csv",
    index=False
)

DEPENDENCY_DF.to_csv(
    DATASET_PROFILE_DIR /
    "feature_dependencies.csv",
    index=False
)

print(
    "Dataset-level profile tables saved."
)

Dataset-level profile tables saved.


In [55]:
# ============================================================
# 05.28 SAVE FEATURE PROFILES
# ============================================================

FEATURE_TYPE_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_types.csv",
    index=False
)

FEATURE_MISSINGNESS_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_missingness.csv",
    index=False
)

FEATURE_DISTRIBUTION_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_distributions.csv",
    index=False
)

CARDINALITY_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_cardinality.csv",
    index=False
)

FEATURE_DEPENDENCY_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_dependencies.csv",
    index=False
)

# Assign MUTUAL_INFORMATION_DF to FEATURE_PREDICTIVE_RELEVANCE_DF
FEATURE_PREDICTIVE_RELEVANCE_DF = MUTUAL_INFORMATION_DF

FEATURE_PREDICTIVE_RELEVANCE_DF.to_csv(
    FEATURE_PROFILE_DIR /
    "feature_predictive_relevance.csv",
    index=False
)

print(
    "Feature-level profile tables saved."
)

Feature-level profile tables saved.


In [56]:
# ============================================================
# 05.29 SAVE AIR-LLM JSON CONTEXT
# ============================================================

def make_json_safe(obj):

    if isinstance(
        obj,
        dict
    ):

        return {
            str(k):
                make_json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(
        obj,
        list
    ):

        return [
            make_json_safe(v)
            for v in obj
        ]

    if isinstance(
        obj,
        tuple
    ):

        return [
            make_json_safe(v)
            for v in obj
        ]

    if isinstance(
        obj,
        (np.integer,)
    ):

        return int(obj)

    if isinstance(
        obj,
        (np.floating,)
    ):

        value = float(obj)

        return (
            None
            if not np.isfinite(value)
            else value
        )

    if isinstance(
        obj,
        (np.bool_,)
    ):

        return bool(obj)

    if isinstance(
        obj,
        float
    ):

        return (
            None
            if not np.isfinite(obj)
            else obj
        )

    return obj


DATASET_CONTEXT_PATH = (
    PROFILE_ROOT /
    "dataset_profiles.json"
)

FEATURE_CONTEXT_PATH = (
    PROFILE_ROOT /
    "feature_profiles.json"
)

AIR_LLM_CONTEXT_PATH = (
    PROFILE_ROOT /
    "air_llm_context.json"
)

with open(
    DATASET_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            DATASET_PROFILES
        ),
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    FEATURE_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            FEATURE_PROFILES
        ),
        f,
        indent=2,
        ensure_ascii=False
    )

with open(
    AIR_LLM_CONTEXT_PATH,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        make_json_safe(
            COMPACT_AIR_LLM_CONTEXT
        ),
        f,
        indent=2,
        ensure_ascii=False
    )

print("=" * 90)
print("AIR-LLM CONTEXT FILES SAVED")
print("=" * 90)

print(
    DATASET_CONTEXT_PATH
)

print(
    FEATURE_CONTEXT_PATH
)

print(
    AIR_LLM_CONTEXT_PATH
)

AIR-LLM CONTEXT FILES SAVED
/content/drive/MyDrive/AIR_LLM_Research/results/profiling/dataset_profiles.json
/content/drive/MyDrive/AIR_LLM_Research/results/profiling/feature_profiles.json
/content/drive/MyDrive/AIR_LLM_Research/results/profiling/air_llm_context.json


In [57]:
# ============================================================
# 05.30 FINAL NOTEBOOK 05 VERIFICATION
# ============================================================

print("=" * 90)
print("FINAL NOTEBOOK 05 VERIFICATION")
print("=" * 90)

# ------------------------------------------------------------
# Dataset profiles
# ------------------------------------------------------------

assert len(
    DATASET_PROFILES
) == len(
    DATASET_IDS
)

# ------------------------------------------------------------
# Feature profiles
# ------------------------------------------------------------

for dataset_id in DATASET_IDS:

    assert (
        dataset_id
        in FEATURE_PROFILES
    )

    assert len(
        FEATURE_PROFILES[
            dataset_id
        ]
    ) > 0

# ------------------------------------------------------------
# Required feature profile components
# ------------------------------------------------------------

REQUIRED_COMPONENTS = {
    "T",
    "M",
    "D",
    "R",
    "C",
    "Y"
}

for dataset_id in DATASET_IDS:

    for feature, profile in (
        FEATURE_PROFILES[
            dataset_id
        ].items()
    ):

        assert REQUIRED_COMPONENTS.issubset(
            set(profile.keys())
        )

# ------------------------------------------------------------
# Saved files
# ------------------------------------------------------------

REQUIRED_FILES = [

    DATASET_CONTEXT_PATH,

    FEATURE_CONTEXT_PATH,

    AIR_LLM_CONTEXT_PATH,

    DATASET_PROFILE_DIR /
    "dataset_size.csv",

    DATASET_PROFILE_DIR /
    "missingness_statistics.csv",

    DATASET_PROFILE_DIR /
    "distribution_statistics.csv",

    DATASET_PROFILE_DIR /
    "outlier_statistics.csv",

    FEATURE_PROFILE_DIR /
    "feature_types.csv",

    FEATURE_PROFILE_DIR /
    "feature_missingness.csv",

    FEATURE_PROFILE_DIR /
    "feature_distributions.csv",

    FEATURE_PROFILE_DIR /
    "feature_cardinality.csv",

    FEATURE_PROFILE_DIR /
    "feature_dependencies.csv",

    FEATURE_PROFILE_DIR /
    "feature_predictive_relevance.csv"
]

for path in REQUIRED_FILES:

    assert path.exists(), (
        f"Required output missing:\n{path}"
    )

# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

print()

for dataset_id in DATASET_IDS:

    n_features = len(
        FEATURE_PROFILES[
            dataset_id
        ]
    )

    n_incomplete = len(
        COMPACT_AIR_LLM_CONTEXT[
            dataset_id
        ][
            "incomplete_features"
        ]
    )

    print(
        f"{dataset_id:20s} "
        f"Features: {n_features:3d} | "
        f"Incomplete: {n_incomplete:3d}"
    )

print()

print("=" * 90)
print("NOTEBOOK 05 COMPLETED SUCCESSFULLY")
print("=" * 90)

print(
    "Dataset profiles constructed."
)

print(
    "Feature profiles constructed."
)

print(
    "AIR-LLM context constructed."
)

print(
    "All profiling artifacts saved."
)

print(
    "Ready for AIR-LLM strategy reasoning."
)

FINAL NOTEBOOK 05 VERIFICATION

adult_income         Features:  15 | Incomplete:   0
bank_marketing       Features:  17 | Incomplete:   0
diabetes_130us       Features:  48 | Incomplete:   9

NOTEBOOK 05 COMPLETED SUCCESSFULLY
Dataset profiles constructed.
Feature profiles constructed.
AIR-LLM context constructed.
All profiling artifacts saved.
Ready for AIR-LLM strategy reasoning.
